### Статистический анализ скидочной политики и прибыльности розничной сети «Соседушка»

**Бизнес-задача:** исследовать взаимосвязь скидок, размера покупательской корзины и валовой прибыли розничной сети.

**Цель:** выявить статистически обоснованные различия и закономерности, которые могут быть использованы для оценки скидочной политики.

**Данные:** синтетический датасет розничных продаж за 2024–2025 годы.

**Методы:** описательная статистика, визуализация распределений, доверительные интервалы, статистические критерии и оценка размеров эффекта.

In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import scipy
import statsmodels
import matplotlib
import seaborn as sns

In [3]:
DATA_PATH = Path("../data/raw/simulative_retail_sales.xlsx")

assert DATA_PATH.exists(), f"Файл не найден: {DATA_PATH.resolve()}"

excel = pd.ExcelFile(DATA_PATH)

print("Листы Excel:")
for sheet in excel.sheet_names:
    print(f"- {sheet}")

Листы Excel:
- README
- fact_sales
- dim_products
- dim_stores
- dim_customers
- dim_suppliers


#### 1.2. Загрузка и первичный анализ структуры данных
На этом этапе проверяем состав исходных таблиц, количество наблюдений, названия столбцов и типы данных.

**Цель проверки** — определить структуру датасета, выявить потенциальные проблемы с типами данных и подготовить основу для дальнейшего анализа качества.

In [ ]:
# Загружаем таблицы в словарь DataFrame
sheet_names = [
    'fact_sales',
    'dim_products',
    'dim_stores',
    'dim_customers',
    'dim_suppliers'
]

tables = {
    sheet: pd.read_excel(DATA_PATH, sheet_name=sheet)
    for sheet in sheet_names
}

# Формируем сводную информацию
structure_summary = pd.DataFrame([
    {
        "Таблица": name,
        "Строк": df.shape[0],
        "Столбцов": df.shape[1]
    }
    for name, df in tables.items()
])

display(structure_summary)

,Таблица,Строк,Столбцов
0,fact_sales,81238,10
1,dim_products,101,8
2,dim_stores,20,7
3,dim_customers,5000,7
4,dim_suppliers,20,4


#### Шаг 1.3. Проверяем названия столбцов и типы данных

In [9]:
for name, df in tables.items():
    print(f"\n{'=' * 60}")
    print(f"Таблица: {name}")
    print(f"{'=' * 60}")

    df.info()
    print("\nНазвания столбцов:")
    print(df.columns.tolist())


Таблица: fact_sales
<class 'pandas.DataFrame'>
RangeIndex: 81238 entries, 0 to 81237
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id_чека          81238 non-null  int64         
 1   дата_время       81238 non-null  datetime64[us]
 2   id_магазина      81238 non-null  int64         
 3   id_товара        81238 non-null  int64         
 4   id_клиента       44744 non-null  float64       
 5   количество       81238 non-null  int64         
 6   цена_за_единицу  81238 non-null  float64       
 7   скидка_процент   81238 non-null  int64         
 8   сумма_строки     81238 non-null  float64       
 9   способ_оплаты    81238 non-null  str           
dtypes: datetime64[us](1), float64(3), int64(5), str(1)
memory usage: 6.2 MB

Названия столбцов:
['id_чека', 'дата_время', 'id_магазина', 'id_товара', 'id_клиента', 'количество', 'цена_за_единицу', 'скидка_процент', 'сумма_строки', 'способ_оплат

#### 1.4. Проверка гранулярности и уникальности ключей
Перед статистическим анализом необходимо определить единицу наблюдения и проверить уникальность идентификаторов в справочниках.

Дублирование ключей может привести к размножению строк при объединении таблиц и, как следствие, к искажению финансовых показателей.

In [11]:
# Проверяем уникальность ключей справочников
dimension_keys = {
    'dim_products': 'id_товара',
    'dim_stores': 'id_магазина',
    'dim_customers': 'id_клиента',
    'dim_suppliers': 'id_поставщика'
}

key_checks = []

for table_name, key in dimension_keys.items():
    df = tables[table_name]

    key_checks.append({
        "Таблица": table_name,
        "Ключ": key,
        "Строк": len(df),
        "Уникальных ключей": df[key].nunique(),
        "Дубликатов": df[key].duplicated().sum(),
        "Пропусков": df[key].isna().sum()
    })

    display(pd.DataFrame(key_checks))

,Таблица,Ключ,Строк,Уникальных ключей,Дубликатов,Пропусков
0,dim_products,id_товара,101,101,0,0


,Таблица,Ключ,Строк,Уникальных ключей,Дубликатов,Пропусков
0,dim_products,id_товара,101,101,0,0
1,dim_stores,id_магазина,20,20,0,0


,Таблица,Ключ,Строк,Уникальных ключей,Дубликатов,Пропусков
0,dim_products,id_товара,101,101,0,0
1,dim_stores,id_магазина,20,20,0,0
2,dim_customers,id_клиента,5000,5000,0,0


,Таблица,Ключ,Строк,Уникальных ключей,Дубликатов,Пропусков
0,dim_products,id_товара,101,101,0,0
1,dim_stores,id_магазина,20,20,0,0
2,dim_customers,id_клиента,5000,5000,0,0
3,dim_suppliers,id_поставщика,20,20,0,0


In [12]:
# Проверяем структуру чеков
sales = tables["fact_sales"]

check_summary = pd.DataFrame({
    "Показатель": [
        "Количество строк",
        "Количество уникальных чеков",
        "Среднее количество строк в чеке",
        "Максимальное количество строк в чеке",
        "Полностью дублирующиеся строки"
    ],
    "Значение": [
        len(sales),
        sales["id_чека"].nunique(),
        len(sales) / sales["id_чека"].nunique(),
        sales.groupby("id_чека").size().max(),
        sales.duplicated().sum()
    ]
})

display(check_summary)

,Показатель,Значение
0,Количество строк,81238.000000
1,Количество уникальных чеков,22790.000000
2,Среднее количество строк в чеке,3.564634
3,Максимальное количество строк в чеке,12.000000
4,Полностью дублирующиеся строки,0.000000


In [13]:
# Проверяем согласованность данных внутри чека
check_consistency = (
    sales.groupby("id_чека")
    .agg(
        stores=("id_магазина", "nunique"),
        timestamps=("дата_время", "nunique"),
        customers=("id_клиента", lambda x: x.nunique(dropna=False)),
        payments=("способ_оплаты", "nunique")
    )
)

consistency_result = pd.DataFrame({
    "Проверка": [
        "Чек связан с несколькими магазинами",
        "Чек содержит разные даты/время",
        "Чек связан с разными покупателями",
        "Чек содержит разные способы оплаты"
    ],
    "Количество проблемных чеков": [
        (check_consistency["stores"] > 1).sum(),
        (check_consistency["timestamps"] > 1).sum(),
        (check_consistency["customers"] > 1).sum(),
        (check_consistency["payments"] > 1).sum()
    ]
})

display(consistency_result)

,Проверка,Количество проблемных чеков
0,Чек связан с несколькими магазинами,0
1,Чек содержит разные даты/время,0
2,Чек связан с разными покупателями,0
3,Чек содержит разные способы оплаты,0


#### Вывод по проверке ключей и гранулярности

Проверена уникальность идентификаторов четырёх справочников. Дублирующиеся и отсутствующие значения первичных ключей не обнаружены.

Таблица продаж содержит 81 238 строк, объединённых в 22 790 уникальных чеков. Среднее количество товарных позиций на чек составляет 3,56, максимальное — 12.

Полных дубликатов строк не обнаружено. Все чеки согласованы по магазину, дате и времени, покупателю и способу оплаты.

**Единица наблюдения:** исходная строка соответствует товарной позиции продажи; для анализа среднего чека, размера корзины и валовой прибыли на покупку потребуется агрегация до уровня чека.

Следующий этап контроля качества — проверка соответствия внешних ключей справочникам, пропусков и допустимых диапазонов значений.

#### 1.5. Проверка ссылочной целостности

In [14]:
foreign_keys = [
    ("fact_sales", "id_товара", "dim_products", "id_товара"),
    ("fact_sales", "id_магазина", "dim_stores", "id_магазина"),
    ("fact_sales", "id_клиента", "dim_customers", "id_клиента"),
    ("dim_products", "id_поставщика", "dim_suppliers", "id_поставщика")
]

fk_results = []

for fact_name, fact_key, dim_name, dim_key in foreign_keys:
    fact = tables[fact_name]
    dim = tables[dim_name]

    # Для клиентов отсутствие ID допускается
    values = fact[fact_key].dropna()

    missing_mask = ~values.isin(dim[dim_key])

    fk_results.append({
        "Таблица": fact_name,
        "Внешний ключ": fact_key,
        "Справочник": dim_name,
        "Несопоставленных строк": missing_mask.sum(),
        "Уникальных отсутствующих ID": values[missing_mask].nunique()
    })

display(pd.DataFrame(fk_results))

,Таблица,Внешний ключ,Справочник,Несопоставленных строк,Уникальных отсутствующих ID
0,fact_sales,id_товара,dim_products,0,0
1,fact_sales,id_магазина,dim_stores,0,0
2,fact_sales,id_клиента,dim_customers,0,0
3,dim_products,id_поставщика,dim_suppliers,0,0


#### Шаг 1.6. Проверка пропусков

In [15]:
missing_results = []

for name, df in tables.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()

        if missing_count > 0:
            missing_results.append({
                "Таблица": name,
                "Столбец": column,
                "Пропусков": missing_count,
                "Доля пропусков, %": round(
                    missing_count / len(df) * 100, 2
                )
            })

display(pd.DataFrame(missing_results))

,Таблица,Столбец,Пропусков,"Доля пропусков, %"
0,fact_sales,id_клиента,36494,44.92


#### Шаг 1.7. Проверка допустимых значений

In [16]:
sales = tables["fact_sales"]

value_checks = pd.DataFrame({
    "Проверка": [
        "Количество <= 0",
        "Цена за единицу <= 0",
        "Скидка < 0 или > 100%",
        "Сумма строки < 0",
        "Сумма строки = 0"
    ],
    "Количество строк": [
        (sales["количество"] <= 0).sum(),
        (sales["цена_за_единицу"] <= 0).sum(),
        (
            (sales["скидка_процент"] < 0) |
            (sales["скидка_процент"] > 100)
        ).sum(),
        (sales["сумма_строки"] < 0).sum(),
        (sales["сумма_строки"] == 0).sum()
    ]
})

display(value_checks)

print(
    "Уровни скидок:",
    sorted(sales["скидка_процент"].unique())
)

print(
    "Способы оплаты:",
    sales["способ_оплаты"].unique()
)

,Проверка,Количество строк
0,Количество <= 0,0
1,Цена за единицу <= 0,0
2,Скидка < 0 или > 100%,0
3,Сумма строки < 0,0
4,Сумма строки = 0,0


Уровни скидок: [np.int64(0), np.int64(5), np.int64(10), np.int64(15)]
Способы оплаты: <StringArray>
['карта', 'СБП', 'наличные']
Length: 3, dtype: str


#### Вывод по проверке ссылочной целостности, пропусков и допустимых значений

Проверка ссылочной целостности не выявила несоответствий между таблицей продаж и справочниками товаров, магазинов, покупателей и поставщиков.

Пропущенные значения обнаружены только в поле `id_клиента`: 36 494 строки (44,92%). Предположительно, пропуски связаны с покупками без идентификации клиента в программе лояльности. На текущем этапе строки сохранены.

В количественных полях не выявлено нулевых или отрицательных значений количества, цены и суммы продажи. Скидки находятся в допустимом диапазоне и принимают значения 0%, 5%, 10% и 15%.

Проверенные структурные и диапазонные характеристики не содержат нарушений. Для завершения аудита необходимо проверить корректность расчёта сумм продаж и бизнес-правила применения скидок.

#### Шаг 1.8. Проверка финансовых расчётов

In [17]:
# Рассчитываем ожидаемую сумму продажи
expected_amount = (
    sales["количество"]
    * sales["цена_за_единицу"]
    * (1 - sales["скидка_процент"] / 100)
)

# Рассчитываем абсолютное расхождение
amount_diff = (sales["сумма_строки"] - expected_amount).abs()

# Допуск 1 копейка на округление
tolerance = 0.01

financial_checks = pd.DataFrame({
    "Показатель": [
        "Количество проверенных строк",
        "Строк с расхождением более 1 копейки",
        "Максимальное расхождение, руб.",
        "Среднее расхождение, руб."
    ],
    "Значение": [
        len(sales),
        (amount_diff > tolerance).sum(),
        amount_diff.max(),
        amount_diff.mean()
    ]
})

display(financial_checks)

,Показатель,Значение
0,Количество проверенных строк,81238.000000
1,Строк с расхождением более 1 копейки,0.000000
2,"Максимальное расхождение, руб.",0.005000
3,"Среднее расхождение, руб.",0.000491


#### Шаг 1.9. Проверка бизнес-правил скидок

In [18]:
# Признак наличия идентификатора клиента
sales["has_customer_id"] = sales["id_клиента"].notna()

# Признак применения скидки
sales["has_discount"] = sales["скидка_процент"] > 0

# Таблица распределения товарных позиций
discount_customer_check = pd.crosstab(
    sales["has_customer_id"],
    sales["has_discount"],
    margins=True
)

display(discount_customer_check)

has_discount,False,True,All
has_customer_id,,,
False,36494,0,36494
True,28911,15833,44744
All,65405,15833,81238


#### Вывод по финансовым расчётам и применению скидок

Проверка 81 238 строк продаж не выявила расхождений между исходной суммой продажи и расчётной величиной более 0,01 ₽. Максимальное абсолютное расхождение составило 0,005 ₽, что согласуется с округлением денежных значений.

Анализ применения скидок показал, что все 15 833 товарные позиции со скидкой связаны с идентифицированными покупателями. Среди позиций с заполненным `id_клиента` доля скидочных продаж составляет 35,39%.

Полученная закономерность согласуется с предполагаемым правилом предоставления скидок участникам программы лояльности.

**Методологическое следствие:** при проверке различий между покупками со скидкой и без скидки необходимо учитывать наличие идентификатора клиента. Основное сравнение среднего чека будет проводиться среди идентифицированных покупателей.

Исходная сумма продажи признана пригодной для дальнейшего анализа выручки.

#### 1.10. Проверка временного охвата данных

In [19]:
# Границы исследуемого периода
start_date = pd.Timestamp("2024-01-01")
end_date = pd.Timestamp("2026-01-01")

date_checks = pd.DataFrame({
    "Показатель": [
        "Минимальная дата продажи",
        "Максимальная дата продажи",
        "Продаж вне периода 2024–2025",
        "Количество месяцев с продажами",
        "Продаж раньше открытия магазина"
    ],
    "Значение": [
        sales["дата_время"].min(),
        sales["дата_время"].max(),
        (
            (sales["дата_время"] < start_date) |
            (sales["дата_время"] >= end_date)
        ).sum(),
        sales["дата_время"].dt.to_period("M").nunique(),
        (
            sales.merge(
                tables["dim_stores"][["id_магазина", "дата_открытия"]],
                on="id_магазина",
                how="left",
                validate="many_to_one"
            )
            .eval("дата_время < дата_открытия")
            .sum()
        )
    ]
})

display(date_checks)

,Показатель,Значение
0,Минимальная дата продажи,2024-01-01 09:42:00
1,Максимальная дата продажи,2025-12-31 22:40:00
2,Продаж вне периода 2024–2025,0
3,Количество месяцев с продажами,24
4,Продаж раньше открытия магазина,0


#### 1.11. Итоги аудита качества данных

Проведён аудит исходного набора данных розничной сети «Соседушка», включающего 81 238 товарных позиций, 22 790 чеков и справочники товаров, магазинов, покупателей и поставщиков.

**Результаты проверки:**

- Первичные ключи справочников уникальны, пропуски и дубликаты отсутствуют.
- Согласованность чеков по магазину, дате, покупателю и способу оплаты подтверждена.
- Нарушений ссылочной целостности не обнаружено.
- Некорректных значений количества, цены, суммы продажи и процента скидки не выявлено.
- Расхождения между исходными и расчётными суммами продаж не превышают 0,005 ₽.
- Продажи охватывают 24 месяца — с января 2024 по декабрь 2025 года. Продаж до открытия магазинов не обнаружено.
- Пропуски присутствуют только в поле `id_клиента` (44,92% строк). Все позиции со скидкой связаны с идентифицированными покупателями.

**Заключение:** критических нарушений качества данных по выполненным проверкам не выявлено. Исходные наблюдения сохраняются без исключений.

Для дальнейшего анализа будут сформированы две аналитические выборки:

1. На уровне товарной позиции — для изучения скидок, категорий и маржинальности.
2. На уровне чека — для анализа размера покупательской корзины, среднего чека и валовой прибыли на покупку.

При интерпретации результатов необходимо учитывать синтетическое происхождение данных и ограничения расчётной себестоимости.